# **Task 2 — DeiT**: does seeing more pixels help?

This week's seminar was about [DeiT](https://proceedings.mlr.press/v139/touvron21a.html): a ViT trained on ImageNet alone, with distillation, and fine-tuned at a higher resolution at the end. The paper reports that going from 224×224 to 384×384 images makes the model more accurate (Table 10). Today you check whether that holds for **our** model and **our** data.

This notebook loads the data and the model and gives you a **fixed evaluation**, so that every group measures its result the same way. Your own work starts after the baseline.

<details>
<summary>Reach out here if you have any problems</summary>
Taras Kutsyk — taras.kutsyk@doctoral.uj.edu.pl
</details>

# The task

**Question.** Does seeing more pixels help? Make the model run at 384×384, fine-tune it within the budget, and **answer the question for this model and this data, with evidence**. Then explain *why* you got the result you got, and what they teach you.

*Hint:* be wary of what you compare with, and of how your claim relates to the paper's.

**Fixed:**
- the starting model and the dataset;
- the `evaluate()` function below;
- the fine-tuning budget: **at most 100,000 training images per fine-tuned model** (about 11 passes over the training set);
- write the resizing of the position embeddings yourselves: no ready-made resizing (e.g. timm's `img_size=` argument or `resample_abs_pos_embed`).

Everything else is your choice.

**Bring to the lab:** your code + a short deck with your answer, the evidence for it, and your explanation. If your group defends this week, **every member should be able to explain any piece of the code**.

# Setup

**Step 0.** Runtime → Change runtime type → **T4 GPU**. Everything we need (PyTorch, torchvision, timm) is already installed on Colab.

**Step 1. Data.** [Imagewoof](https://github.com/fastai/imagenette): 10 dog breeds from ImageNet, 9,025 training and 3,929 validation photos, **at full size** (about 1.3 GB, under a minute to download).

In [ ]:
import torch, timm

# downloads Imagewoof if missing, builds `cfg` and `to_imagenet`, defines the fixed `evaluate()`
from common import cfg, device, evaluate, to_imagenet

**Step 2. Model.** A distilled DeiT-S from the paper (12 blocks, 384 dims, 22M parameters, the DeiT-S⚗ of Table 5), trained on ImageNet at 224×224, together with the exact preprocessing it was trained with.

The model predicts one of 1000 ImageNet classes, so we map each Imagewoof folder (e.g. `n02086240` = Shih-Tzu) to its **ImageNet index**: the position of that breed among the model's 1000 outputs. You will need `to_imagenet` for your fine-tuning too.

<details>
<summary>Running the cell below can ask for your HF_TOKEN secret</summary>
Just press "Grant Access", it's automatically generated by Colab
</details>

In [ ]:
model = (
    timm.create_model("deit_small_distilled_patch16_224", pretrained=True)
    .to(device)
    .eval()
)

**Step 3. The fixed evaluation** (DON'T modify it). `evaluate(model, resolution)` resizes the validation photos for the given resolution exactly as the model's own preprocessing does, and returns the **top-1 accuracy over the 10 breeds**: the model's prediction is the highest-scoring of its 10 dog outputs.

In [ ]:
# `evaluate` is imported from common.py above (unchanged from the starter), so every experiment uses the same one
help(evaluate)

# 0-shot @384 per interpolation mode

**Author: Piotr** (`exp_a_zero_shot.py`, pos-embed resizing in `common.py`)

In [ ]:
from exp_a_zero_shot import MODES, run_zero_shot

SMALLER_RES = 224
BIGGER_RES = 384

chosen_resolution = SMALLER_RES

In [ ]:
# 0-shot @384 for every interpolation mode; each run resizes its own copy of `model`
zero_shot = {mode: run_zero_shot(model, BIGGER_RES, mode)[1] for mode in MODES}

**Step 4. Baseline.** Takes about half a minute. You should get **top-1 ≈ 97.20%** at 224. If not, something in the setup went wrong.

# Fine-tuning control @224

**Author: Piotr** (`finetune_pre_resizing.py`)

In [ ]:
from finetune_pre_resizing import run_control

In [ ]:
model_control, acc_control = run_control(model, chosen_resolution)

# Before you start

Think about **what exactly breaks in the model when you pass it higher-resolution images?** What's the simplest, most effective way to fix that?

Training photos are in `imagewoof2/train` (same folder structure as `val`).

# Your solution

Make the model run at 384, fine-tune within the budget, and collect the evidence for your answer. Good luck!

In [ ]:
# your_model = ...
# print(f"ours @384: top-1 {evaluate(your_model, 384):.2%}")

# Linear probe (B1, B2) and layer freezing (D) @384

**Author: Gabriela**

Definitions are in `exp_b_linear_probe.py` and `exp_d_layer_freezing.py`. Each run trains its own
copy of the model it is given, so `model` stays the untouched pretrained DeiT-S.

In [ ]:
from exp_b_linear_probe import run_b1, run_b2
from exp_d_layer_freezing import run_d

In [ ]:
model_b1, acc_b1 = run_b1(model)

In [ ]:
model_b2, acc_b2 = run_b2(model)

In [ ]:
model_d, acc_d = run_d(model)

# Full fine-tuning (C) and progressive resolution (E) @384

**Author: Vitaliy**

Definitions are in `exp_c_full_finetune.py` and `exp_e_progressive.py`. Both train their own copy
of `model`, and load a saved checkpoint instead of retraining if one exists.

In [ ]:
from exp_c_full_finetune import run_c
from exp_e_progressive import run_e

In [ ]:
model_c, acc_c = run_c(model)

In [ ]:
model_e, acc_e = run_e(model)